# We will process and merger the may data here

In [1]:
import polars as pl
from pathlib import Path
import unicodedata
import re
import os
import shutil
from time import sleep
from datetime import datetime

def clean_folder(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
    os.makedirs(path, exist_ok=True)

def clean_column(name):
    replacements = {
        'µ': 'u',
        'μ': 'u',
        '²': '2',
        '³': '3',
    }
    
    for char, replacement in replacements.items():
        name = name.replace(char, replacement)
    
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("ascii")
    name = name.strip()
    name = name.strip('_')
    
    return name

def parse_harmony_dataset(root: Path) -> Path:
    """
    Parses a harmony file directory and yields the cell analysis file

    Args:
        root (Path): Top level folder where all the harmony datasets are located

    Yields:
        Path, str: yields a file path to the cell analysis AND the harmony dataset name
    
    **NOTE:** The harmony file structure is:
        
        raw name/
        ├─ Evaluation/
        │  ├─ PlateResult.txt
        │  ├─ Objects_Population - NoBorderObj.txt (this is analysis file)
        │  ├─ .ExportDone_CellListsToTxt
        ├─ indexfile.txt

    """
    
    for f in root.glob("*/*/*"):
        if f.is_file() and not f.name.startswith(".") and f.name != "PlateResults.txt":
            # print(f)
            yield f, f.relative_to(root).parts[0]


In [2]:
in_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                 "Study/May-Harmony-datasets/")

out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/formatted_raw_datasets/2026-05-11_IMAGE-ISSUES")

# print(list(parse_harmony_dataset(test_path)))
for f_path, ds_name in parse_harmony_dataset(in_path):
    print(ds_name)
    
    out_name = out_path/f"{ds_name}.tsv"
    with (open(f_path, "r", encoding="utf-8") as in_file, 
          open(out_name, "w", encoding='utf-8') as out_file):
        for i, line in enumerate(in_file):
            curr_line = line.strip()
            
            if i < 9:
                print(f"{curr_line} :: {i}")
                continue
            if i == 9:
                cleaned = "\t".join(clean_column(col) for col in line.strip().split("\t"))
                out_file.write(cleaned + "\n")
            else:
                out_file.write(line.rstrip("\t\n") + "\n") 
            
    print("\n\n")
    

# Let's examine the data now

In [3]:
drop_cols = [
    "Row",
    "Column",
    "Field",
    "Plane",
    "Object No",
    "X",
    "Y",
    "Bounding Box",
    "Position X [um]",
    "Position Y [um]",
    "Compound",
    "Concentration",
    "Cell Type",
    "Cell Count"
]

def clean_folder(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
    os.makedirs(path, exist_ok=True)

In [ ]:
input_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                  "Study/formatted_raw_datasets/2026-05-11_IMAGE-ISSUES"
                  "")

out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data/2026-05-11_IMAGE-ISSUES")

clean_folder(out_path)

# out_path = out_path / input_path.name

# NOTE: Need to shift over by -1 in char()

for f in input_path.iterdir():
    print(f.name)
    
    lf = pl.scan_csv(f, separator='\t', null_values=["", "NA", "NaN", "nan", "NULL"])
    
    lf = (lf
          .with_columns(
            (
                pl.col("Row").map_elements(lambda x: chr(64 + x), return_dtype=pl.String) +
                pl.col("Column").cast(pl.String)
            ).alias("WellName")
          )
          .select(
              ["WellName", pl.all().exclude('WellName')]
          )
          .drop(drop_cols)
    )
    
    out_file = out_path / f.name
    lf.sink_csv(out_file, separator="\t")

20260507-Measurement 2.tsv
20260511-Measurement 1.tsv


## Merge and stitch the datasets together

In [5]:
ds1_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data/2026-05-11_IMAGE-ISSUES/20"
                "260507-Measurement 2.tsv")

test_lf = pl.scan_csv(ds1_path, separator="\t")
display(list(test_lf.collect_schema().to_python())[:25])

['WellName',
 'Timepoint',
 'NoBorderNuc - ROI No',
 'NoBorderNuc - Nucleus Area [um2]',
 'NoBorderNuc - Nucleus Roundness',
 'NoBorderNuc - Nucleus Perimeter [um]',
 'NoBorderNuc - Nucleus Width [um]',
 'NoBorderNuc - Nucleus Length [um]',
 'NoBorderNuc - Nucleus Ratio Width to Length',
 'NoBorderNuc - IntensityNucHOECHST Mean',
 'NoBorderNuc - Object No in Nuclei',
 'NoBorderNuc - Object No in GoodNuclei',
 'NoBorderNuc - Intensity Nucleus Region HOECHST 33342 narrow Mean',
 'NoBorderNuc - Intensity Nucleus Region HOECHST 33342 narrow StdDev',
 'NoBorderNuc - Intensity Nucleus Region HOECHST 33342 narrow Median',
 'NoBorderNuc - Intensity Nucleus Region HOECHST 33342 narrow Maximum',
 'NoBorderNuc - Intensity Nucleus Region HOECHST 33342 narrow Minimum',
 'NoBorderNuc - Intensity Nucleus Region HOECHST 33342 narrow Sum',
 'NoBorderNuc - Intensity Nucleus Region HOECHST 33342 narrow CV [%]',
 'NoBorderNuc - Intensity Nucleus Region HOECHST 33342 narrow Quantile 90%',
 'NoBorderNuc - I

In [2]:
ds1_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data/2026-05-11_IMAGE-ISSUES/20"
                "260507-Measurement 2.tsv")

ds2_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data/2026-05-11_IMAGE-ISSUES/20"
                "260511-Measurement 1.tsv")

sampling_interval = 1.5
timepoint_col = "Timepoint"

t0_ds1 = datetime(2026, 5, 7, 19, 18)
t0_ds2 = datetime(2026, 5, 11, 14, 42)

offset_hours_raw = (t0_ds2 - t0_ds1).total_seconds() / 3600
offset_hours = round(offset_hours_raw * 2) / 2
print(f"Dataset 2 starts {offset_hours_raw:.2f} hours after dataset 1")
print(f"That is {offset_hours_raw / 24:.2f} days.\n")

print(f"To the nearest {sampling_interval}h mark, datset 2 starts {offset_hours:.2f} hours after"
      f" dataset1.")
print(f"This is {offset_hours / 24:.2f} days.\n")

# converting tps

lf1 = (
    pl.scan_csv(ds1_path, separator="\t")
    .with_columns(
        ((pl.col(timepoint_col) - 1) * sampling_interval)
        .alias(timepoint_col)
    )
)

lf2 = (
    pl.scan_csv(ds2_path, separator="\t")
    .with_columns(
        (offset_hours + (pl.col(timepoint_col) - 1) * sampling_interval)
        .alias(timepoint_col)
    )
)

# check gaps

ds1_max = lf1.select(pl.col(timepoint_col).max()).collect().item()

ds2_min = lf2.select(pl.col(timepoint_col).min()).collect().item()
ds2_max = lf2.select(pl.col(timepoint_col).max()).collect().item()

gap_hours = ds2_min - ds1_max
print(f"Dataset 1 ends at: {ds1_max:.1f}h")
print(f"Dataset 2 starts at: {ds2_min:.1f}h")

print(f"The experiment ends at: {ds2_max:.1f}h ({ds2_max/24:.2f} days!)")

print(f"Gap between datasets is: {gap_hours:.1f}h ({gap_hours/24:.2f} days!)")

if gap_hours > sampling_interval:
    print(f"Gap found - {gap_hours:.1f}h of missing data between datasets")
else:
    print("no gaps found")

# combine the datasets together

# concat_lf = pl.concat([lf1,lf2]).sort(['WellName', timepoint_col])
# concat_lf = pl.concat([lf1,lf2])

# out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
#                 "Study/analysis_ready_data")
# out_path = out_path / f"2026-05-11_Harmony_Results._.STITCHED.tsv"

# concat_lf.sink_csv(out_path, separator="\t")

Dataset 2 starts 91.40 hours after dataset 1
That is 3.81 days.

To the nearest 1.5h mark, datset 2 starts 91.50 hours after dataset1.
This is 3.81 days.

Dataset 1 ends at: 90.0h
Dataset 2 starts at: 91.5h
The experiment ends at: 184.5h (7.69 days!)
Gap between datasets is: 1.5h (0.06 days!)
no gaps found


In [3]:
concat_lf = pl.concat([lf1,lf2])

out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data")
out_path = out_path / f"2026-05-11_Harmony_Results._.STITCHED.tsv"

concat_lf.sink_csv(out_path, separator="\t")

# So it turns out I do not need to merge that second dataset

In [12]:
ds1_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data/2026-05-11_IMAGE-ISSUES/20"
                "260507-Measurement 2.tsv")

sampling_interval = 1.5
timepoint_col = "Timepoint"

lf1 = (
    pl.scan_csv(ds1_path, separator="\t")
    .with_columns(
        ((pl.col(timepoint_col) - 1) * sampling_interval)
        .alias(timepoint_col)
    )
)

# display(lf1.head(10).collect())

ds1_min = lf1.select(pl.col(timepoint_col).min()).collect().item()
ds1_max = lf1.select(pl.col(timepoint_col).max()).collect().item()

print(f"Dataset 1 starts as: {ds1_min:.1f}h and ends at {ds1_max:.1f}h.\n"
      f"That is {ds1_max/24:.2f} days.")


# sinking final dataset
out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data")
out_path = out_path / f"2026-05-11_Harmony_Results._.DATASET1.tsv"

lf1.sink_csv(out_path, separator="\t")

Dataset 1 starts as: 0.0h and ends at 90.0h.
That is 3.75 days.
